In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications import MobileNetV2

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    Dense,
    Dropout,
    GlobalAveragePooling2D
)

from tensorflow.keras.optimizers import Adam

In [ ]:
# Dataset path
dataset_path = "../dataset/train/images"

# Image settings
IMG_SIZE = 224

BATCH_SIZE = 32

EPOCHS = 30

In [ ]:
datagen = ImageDataGenerator(
    
    rescale=1./255,

    validation_split=0.2,

    rotation_range=20,

    zoom_range=0.2,

    width_shift_range=0.2,

    height_shift_range=0.2,

    shear_range=0.2,

    horizontal_flip=True,

    fill_mode='nearest'
)

In [ ]:
train_data = datagen.flow_from_directory(
    
    dataset_path,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="training"
)

In [ ]:
val_data = datagen.flow_from_directory(
    
    dataset_path,

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    subset="validation"
)

In [ ]:
print("Class Labels:")

print(train_data.class_indices)

In [ ]:
base_model = MobileNetV2(
    
    weights='imagenet',

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

In [ ]:
base_model.trainable = False

In [ ]:
model = Sequential([

    base_model,

    GlobalAveragePooling2D(),

    Dense(128, activation='relu'),

    Dropout(0.5),

    Dense(train_data.num_classes, activation='softmax')
])

In [ ]:
model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(

    train_data,

    validation_data=val_data,

    epochs=EPOCHS
)

In [ ]:
import os
os.makedirs("model", exist_ok=True)

In [ ]:
model.save("model/steel_defect_mobilenet.h5")

print("✅ MobileNetV2 Model Saved Successfully!")

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("MobileNetV2 Training vs Validation Accuracy")

plt.legend()

plt.show()

# ==============================
# PRINT FINAL ACCURACY
# ==============================

train_acc = history.history['accuracy'][-1] * 100

val_acc = history.history['val_accuracy'][-1] * 100

print(f"Final Training Accuracy: {train_acc:.2f}%")

print(f"Final Validation Accuracy: {val_acc:.2f}%")

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("model/steel_defect_mobilenet.keras")

print("✅ Model Loaded Successfully")

In [ ]:
class_names = [
    "crazing",
    "inclusion",
    "patches",
    "pitted_surface",
    "rolled_in_scale",
    "scratches"
]

In [ ]:
from tensorflow.keras.preprocessing import image

In [ ]:
img_path = "../dataset/train/images/inclusion/inclusion_1.jpg"

In [ ]:
img = image.load_img(img_path)

plt.imshow(img)

plt.axis("off")

plt.show()

In [ ]:
img = image.load_img(
    
    img_path,

    target_size=(IMG_SIZE, IMG_SIZE)
)

img_array = image.img_to_array(img)

img_array = img_array / 255.0

img_array = np.expand_dims(img_array, axis=0)

print(img_array.shape)

In [ ]:
prediction = model.predict(img_array)

predicted_class = class_names[np.argmax(prediction)]

confidence = np.max(prediction) * 100

print("Prediction:", predicted_class)

print("Confidence:", round(confidence, 2), "%")

In [ ]:
import tensorflow as tf
print(tf.__version__)